# Phase 8 — Causal Regularization One-Fold Pilot

**Objective:** Implement the causal training objective and perform one controlled
fold-0 pilot. Do not launch five-fold causal training.

**Rules:**
- Train only fold 0 with causal regularization.
- Load the baseline fold-0 checkpoint as the starting point (or train from scratch).
- Training donors come only from the training partition.
- No validation, test, or external donor may enter training.
- Use a modest pilot epoch budget (20 epochs max).
- Select pilot hyperparameters from training and validation behavior only.
- Do not tune from the fold-0 test results.
- Record every attempted configuration, including failed ones.
- Monitor: validation AUROC, classification loss, each causal loss, gradient norm,
  prediction entropy, necessity-eligible fraction, sufficient-image divergence,
  background-swap divergence, lesion-removed confidence, GPU memory.
- Compare only against the existing fold-0 baseline.
- BUS-UCLM is frozen external validation — must not influence any choices.

**Phase 8 gate:**
- Causal-loss tests pass.
- The pilot remains numerically stable.
- Checkpoint resume passes.
- Donor isolation passes.
- The five-fold configuration is frozen using training and validation evidence only.
- Pilot results are labelled exploratory.

## 8.0 — Colab bootstrap

Detects Google Colab and clones/pulls the repository. In VS Code, does nothing.

In [ ]:
import os
from pathlib import Path


def is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


REPO_URL = "https://github.com/Sayem7456/CausalMask-XAI.git"
COLAB_TARGET = Path("/content/CausalMask-XAI")

if is_colab():
    print("Detected Google Colab environment.")
    if COLAB_TARGET.exists() and (COLAB_TARGET / "CausalMask-XAI.md").exists():
        print(f"Repository present at {COLAB_TARGET}. Pulling latest...")
        !cd {COLAB_TARGET} && git pull --ff-only
        print("Repository updated to latest commit.")
    else:
        if COLAB_TARGET.exists():
            import shutil
            shutil.rmtree(COLAB_TARGET)
        print(f"Cloning repository from {REPO_URL}...")
        !git clone {REPO_URL} {COLAB_TARGET}
        assert (COLAB_TARGET / "CausalMask-XAI.md").exists(), "Clone failed: marker file missing"
    os.environ["CAUSALMASK_PROJECT_ROOT"] = str(COLAB_TARGET)
    !cd {COLAB_TARGET} && pip install -e .[dev] --quiet 2>&1 | tail -3
    print("Package installed in editable mode.")
else:
    print("Not in Colab — skipping bootstrap.")

## 8.1 — Resolve project root

Resolution order:
1. `CAUSALMASK_PROJECT_ROOT` environment variable
2. Walk up from cwd looking for `CausalMask-XAI.md`
3. Colab fallback `/content/CausalMask-XAI`

In [ ]:
import os
import sys
from pathlib import Path


def _resolve_project_root() -> Path:
    env_root = os.environ.get("CAUSALMASK_PROJECT_ROOT")
    if env_root:
        p = Path(env_root)
        if (p / "CausalMask-XAI.md").exists():
            return p.resolve()
    cwd = Path.cwd()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / "CausalMask-XAI.md").exists():
            return candidate.resolve()
    colab_fallback = Path("/content/CausalMask-XAI")
    if colab_fallback.exists() and (colab_fallback / "CausalMask-XAI.md").exists():
        return colab_fallback.resolve()
    raise RuntimeError(
        "Cannot resolve project root. Set CAUSALMASK_PROJECT_ROOT or run from within the repo."
    )


PROJECT_ROOT = _resolve_project_root()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
assert (PROJECT_ROOT / "CausalMask-XAI.md").exists(), "Marker file missing"

src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
print(f"src dir added to path: {src_dir}")


## 8.2 — Freeze and display active configuration

Every training knob is frozen here before execution. These values are
burned into the `config.resolved.yaml` of the run directory.

BUS-UCLM must not influence any of these choices.

In [ ]:
import json
from datetime import datetime, timezone

import torch

from causalmask.reproducibility import capture_environment, configure_reproducibility

SEED = 42
repro_info = configure_reproducibility(seed=SEED)
env_info = capture_environment(project_root=PROJECT_ROOT)

PILOT_CONFIG = {
    "phase": "08",
    "phase_name": "Causal Regularization One-Fold Pilot",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "backbone": "efficientnet_b0",
    "pretrained": True,
    "pretrained_weight_id": "EfficientNet_B0_Weights.IMAGENET1K_V1",
    "num_classes": 2,
    "binary_classes": ["benign", "malignant"],
    "input_size": [224, 224],
    "pilot_fold": 0,
    "num_epochs": 20,
    "batch_size": 16,
    "gradient_accumulation_steps": 1,

    "optimizer": "adamw",
    "learning_rate": 1e-4,
    "weight_decay": 1e-5,
    "scheduler": "reduce_on_plateau",
    "scheduler_patience": 5,
    "scheduler_factor": 0.5,
    "early_stopping_metric": "val_loss",
    "early_stopping_patience": 10,
    "early_stopping_mode": "min",
    "gradient_clip_val": 1.0,
    "amp_enabled": True,
    "label_smoothing": 0.0,

    "causal_loss": {
        "loss_variant": "full",
        "ce_weight": 1.0,
        "sufficiency_weight": 0.5,
        # NOTE: background swap disabled during training (swapped=None); weight applies in validation only
        "background_weight": 0.5,
        "necessity_weight": 0.5,
        "necessity_margin": 0.1,
        "necessity_warmup_epochs": 3,
        "necessity_confidence_threshold": 0.6,
        "necessity_ramp_epochs": 3,
        "use_detached_teacher": True,
    },

    "counterfactual": {
        "margin_ratio": 0.05,
        "blur_sigma": 20.0,
        "removal_operator": "telea",
        "donor_class": "same",
        "n_donors_per_sample": 1,
        "feathered_blend": True,
        "align_histogram": True,
    },

    "augmentation": {
        "horizontal_flip_prob": 0.5,
        "rotation_degrees": 10.0,
        "affine_translate_max": 0.05,
        "affine_scale_min": 0.95,
        "affine_scale_max": 1.05,
        "gamma_range": [0.9, 1.1],
        "contrast_range": [0.9, 1.1],
        "noise_std": 0.005,
    },

    "classification_threshold_policy": "Youden's J statistic computed from validation predictions only",
    "manifest_version": "v1",
    "split_name": "busi_binary_grouped_5fold_v1",
    "external_datasets": ["bus_uclm"],
    "datasets": {
        "busi": {
            "archive_rel": "data/raw/archives/breast-ultrasound-images-dataset.zip",
            "extract_rel": "data/raw/extracted/busi",
        },
        "bus_uclm": {
            "archive_rel": "data/raw/archives/bus-uclm-breast-ultrasound-dataset.zip",
            "extract_rel": "data/raw/extracted/bus_uclm",
        },
    },
    "experiment_note": (
        "Causal regularization one-fold pilot. Fold 0 only. "
        "Full causal objective with warm-up and confidence-gated necessity. "
        "BUS-UCLM is never loaded. Results labelled exploratory."
    ),
}

print(json.dumps(PILOT_CONFIG, indent=2, default=str))

## 8.3 — Mount Drive & restore Phase 2/3/5 artifacts

Mount Google Drive for persistent storage. Restores manifests, splits,
extracted data, and baseline checkpoints from Drive if missing locally.

In [ ]:
import shutil
import zipfile

MANIFESTS_DIR = PROJECT_ROOT / "data" / "manifests"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
REPORTS_DIR = PROJECT_ROOT / "reports"
PHASES_DIR = PROJECT_ROOT / "artifacts" / "phases"
RUNS_DIR = PROJECT_ROOT / "artifacts" / "runs"
ARCHIVES_DIR = PROJECT_ROOT / "data" / "raw" / "archives"
EXTRACT_DIR = PROJECT_ROOT / "data" / "raw" / "extracted"
RESULTS_DIR = REPORTS_DIR / "results"

for d in [MANIFESTS_DIR, SPLITS_DIR, REPORTS_DIR, PHASES_DIR, RUNS_DIR,
          ARCHIVES_DIR, EXTRACT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Manifests dir:  {MANIFESTS_DIR}")
print(f"Splits dir:     {SPLITS_DIR}")
print(f"Runs dir:       {RUNS_DIR}")

# Mount Google Drive
DRIVE_BASE = None
if is_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/CausalMask-XAI")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted. Artifacts will sync to {DRIVE_BASE}")
else:
    print("Not in Colab — Drive not mounted. Artifacts saved locally only.")


def restore_from_drive(subdir, filename, local_dir):
    if DRIVE_BASE is None:
        return False
    src = DRIVE_BASE / subdir / filename
    dst = local_dir / filename
    if dst.exists():
        return False
    if not src.exists():
        print(f"  [WARN] Not on Drive: {src}")
        return False
    local_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  Restored: {dst}")
    return True


def save_to_drive(src, subdir):
    if DRIVE_BASE is None:
        return False
    dst = DRIVE_BASE / subdir / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return True


def save_dir_to_drive(src_dir, subdir):
    if DRIVE_BASE is None:
        return 0
    dst_base = DRIVE_BASE / subdir / src_dir.name
    count = 0
    for f in src_dir.rglob("*"):
        if f.is_file():
            rel = f.relative_to(src_dir)
            dst = dst_base / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dst)
            count += 1
    if count > 0:
        print(f"  Synced {count} files to Drive: {dst_base}")
    return count


print("\n--- Restoring Phase 2/3 artifacts from Drive ---")
for fname in [
    f"busi_manifest_{PILOT_CONFIG['manifest_version']}.parquet",
    f"bus_uclm_manifest_{PILOT_CONFIG['manifest_version']}.parquet",
]:
    restore_from_drive("manifests", fname, MANIFESTS_DIR)

for fname in ["busi_manifest_v2_grouped.parquet"]:
    restore_from_drive("manifests", fname, MANIFESTS_DIR)

restore_from_drive("splits", f"{PILOT_CONFIG['split_name']}.json", SPLITS_DIR)

# Archives and extracted data
for ds_name, cfg in PILOT_CONFIG.get("datasets", {}).items():
    archive_path = ARCHIVES_DIR / Path(cfg["archive_rel"]).name
    restore_from_drive("archives", archive_path.name, ARCHIVES_DIR)
    extract_path = PROJECT_ROOT / cfg["extract_rel"]
    if not extract_path.exists() or not any(extract_path.iterdir()):
        if archive_path.exists():
            print(f"  {ds_name}: extracting from archive...")
            extract_path.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archive_path, "r") as zf:
                zf.extractall(extract_path)
            print(f"  {ds_name}: extracted to {extract_path}")
        else:
            print(f"  {ds_name}: no archive found at {archive_path}.")
    else:
        print(f"  {ds_name}: extracted data already present at {extract_path}")

print("--- Restoring Phase 5 baseline fold-0 from Drive ---")
BASELINE_RUN_ID = f"baseline_ce_effb0_fold{PILOT_CONFIG['pilot_fold']}_seed{PILOT_CONFIG['seed']}"
if DRIVE_BASE is not None:
    for artifact in ["best.pt", "predictions_test.parquet", "metrics_classification.json", "status.json"]:
        if artifact == "best.pt":
            src_d = DRIVE_BASE / "runs" / BASELINE_RUN_ID / "checkpoints" / artifact
            dst_d = RUNS_DIR / BASELINE_RUN_ID / "checkpoints" / artifact
        else:
            src_d = DRIVE_BASE / "runs" / BASELINE_RUN_ID / artifact
            dst_d = RUNS_DIR / BASELINE_RUN_ID / artifact
        if src_d.exists() and not dst_d.exists():
            dst_d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_d, dst_d)
            print(f"  Restored {BASELINE_RUN_ID}/{artifact} from Drive")
        elif not src_d.exists():
            print(f"  [WARN] Not on Drive: {artifact}")
else:
    print(f"  [SKIP] {BASELINE_RUN_ID} — Drive not mounted")
print("--- Restore complete ---\n")

## 8.4 — Verify split and manifest integrity

Before training, confirm the split digest matches. Check that real
extracted BUSI images are available. If either is missing, the
notebook cannot run a real experiment and is **blocked**.

In [ ]:
import pandas as pd

from causalmask.data.splits import load_split, compute_split_digest, compute_manifest_digest

SPLIT_PATH = SPLITS_DIR / f"{PILOT_CONFIG['split_name']}.json"

V2_PATH = MANIFESTS_DIR / "busi_manifest_v2_grouped.parquet"
V1_PATH = MANIFESTS_DIR / f"busi_manifest_{PILOT_CONFIG['manifest_version']}.parquet"

if V2_PATH.exists():
    MANIFEST_PATH = V2_PATH
    manifest_version = "v2_grouped"
elif V1_PATH.exists():
    MANIFEST_PATH = V1_PATH
    manifest_version = "v1"
else:
    MANIFEST_PATH = None
    manifest_version = None

USE_REAL_DATA = SPLIT_PATH.exists() and MANIFEST_PATH is not None

if USE_REAL_DATA:
    split = load_split(SPLIT_PATH)
    split_digest = compute_split_digest(split)
    stored_digest = split.get("metadata", {}).get("split_digest", "")
    digest_match = split_digest == stored_digest
    print(f"Split loaded: {SPLIT_PATH.name}")
    print(f"  Stored digest:   {stored_digest[:16]}...")
    print(f"  Computed digest: {split_digest[:16]}...")
    print(f"  Digest match: {digest_match}")
    if not digest_match:
        raise RuntimeError("SPLIT DIGEST MISMATCH — do not proceed.")
    print("Split integrity: PASSED")

    manifest_df = pd.read_parquet(MANIFEST_PATH)
    manifest_digest = compute_manifest_digest(manifest_df)
    print(f"Manifest loaded: {len(manifest_df)} samples (version: {manifest_version})")
    print(f"  Manifest digest: {manifest_digest[:16]}...")

    BUSI_EXTRACT = PROJECT_ROOT / PILOT_CONFIG["datasets"]["busi"]["extract_rel"]
    HAS_REAL_IMAGES = BUSI_EXTRACT.exists() and any(BUSI_EXTRACT.iterdir())
    print(f"  Real BUSI images extracted: {HAS_REAL_IMAGES}")
    if not HAS_REAL_IMAGES:
        USE_REAL_DATA = False
else:
    print("Split or manifest not found. Real data NOT available. BLOCKED.")
    split = None
    split_digest = "blocked_no_real_data"
    manifest_digest = "blocked_no_real_data"
    HAS_REAL_IMAGES = False
    manifest_df = None

print(f"\nUSE_REAL_DATA = {USE_REAL_DATA}")

## 8.5 — Define counterfactual generation function

This function runs per-batch during training. It:
1. Computes lesion-sufficient images (blur exterior)
2. Computes lesion-removed images (inpaint lesion)
3. Computes background-swapped images (donor from training partition)

Training donors come ONLY from the training partition.
Donors are selected deterministically per sample.

In [ ]:
import numpy as np
import torch
import cv2

from causalmask.counterfactuals.masks import lesion_plus_margin, MarginConfig
from causalmask.counterfactuals.sufficient import generate_lesion_sufficient, SufficientConfig
from causalmask.counterfactuals.removal import generate_lesion_removed, RemovalConfig, RemovalOperator
from causalmask.counterfactuals.background_swap import generate_background_swap, SwapConfig

CF_CONFIG = PILOT_CONFIG["counterfactual"]

margin_cfg = MarginConfig(margin_ratio=CF_CONFIG["margin_ratio"])
suff_cfg = SufficientConfig(margin_config=margin_cfg, blur_sigma=CF_CONFIG["blur_sigma"],
                            use_feathered_blend=CF_CONFIG["feathered_blend"])

removal_op = RemovalOperator(CF_CONFIG["removal_operator"])
removal_cfg = RemovalConfig(margin_config=margin_cfg, operator=removal_op)

swap_cfg = SwapConfig(margin_config=margin_cfg, donor_class=CF_CONFIG["donor_class"],
                      use_feathered_blend=CF_CONFIG["feathered_blend"],
                      align_histogram=CF_CONFIG["align_histogram"], seed=SEED)


def make_counterfactuals_batch(images_tensor, masks_tensor, labels_tensor):
    """Generate counterfactuals for a training batch.

    Args:
        images_tensor: [B, 3, H, W] float tensor in [0, 1] range.
        masks_tensor: [B, 1, H, W] float tensor in [0, 1] or None.
        labels_tensor: [B] int tensor.

    Returns:
        Dict with 'sufficient', 'removed', 'swapped' tensors (same shape as input).
    """
    B = images_tensor.size(0)
    device = images_tensor.device

    sufficient_list = []
    removed_list = []
    swapped_list = []

    for i in range(B):
        img = (images_tensor[i].cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)

        if masks_tensor is not None and masks_tensor[i] is not None:
            msk = (masks_tensor[i].cpu().squeeze(0).numpy() * 255).astype(np.uint8)
        else:
            msk = np.ones(img.shape[:2], dtype=np.uint8) * 128

        suff_img, _ = generate_lesion_sufficient(img, msk, suff_cfg)
        rem_img, _ = generate_lesion_removed(img, msk, removal_cfg)

        suff_t = torch.from_numpy(suff_img).float().permute(2, 0, 1) / 255.0
        rem_t = torch.from_numpy(rem_img).float().permute(2, 0, 1) / 255.0

        sufficient_list.append(suff_t)
        removed_list.append(rem_t)

    result = {
        "sufficient": torch.stack(sufficient_list).to(device),
        "removed": torch.stack(removed_list).to(device),
        "swapped": None,
    }
    return result


print("Counterfactual configuration:")
for k, v in CF_CONFIG.items():
    print(f"  {k}: {v}")
print("Counterfactual generators defined.")
print("\nNOTE: Background swaps are DISABLED during training (swapped=None).")
print("Swap consistency is measured only during validation. See reports/deviations.md.")

## 8.6 — Donor isolation check

Verify that the causal trainer only uses training-partition data.
This cell validates: no validation, test, or external donors can
enter the training loop.

**Note:** The in-batch counterfactual function above does NOT do
background swaps (swapped=None) to keep the pilot simple and avoid
donor-selection complexity during backpropagation. Background swaps
are computed only during validation. This is a documented deviation
for the pilot — see reports/deviations.md.

In [ ]:
# Verify donor isolation: when real data is available, confirm
# that the training dataloader only yields training samples.
if USE_REAL_DATA and split is not None:
    fold_key = f"fold_{PILOT_CONFIG['pilot_fold']}"
    fold_data = split["folds"][fold_key]
    train_ids = set(fold_data["train"])
    val_ids = set(fold_data["validation"])
    test_ids = set(fold_data["test"])

    # Disjointness check
    assert len(train_ids & val_ids) == 0, "Train/val overlap detected"
    assert len(train_ids & test_ids) == 0, "Train/test overlap detected"
    assert len(val_ids & test_ids) == 0, "Val/test overlap detected"

    # No external samples
    external_mask = manifest_df["dataset"] == "bus_uclm"
    external_ids = set(manifest_df[external_mask]["sample_id"].tolist())
    assert len(train_ids & external_ids) == 0, "External samples in training"

    print(f"Donor isolation PASSED.")
    print(f"  Training samples: {len(train_ids)}")
    print(f"  Validation samples: {len(val_ids)}")
    print(f"  Test samples: {len(test_ids)}")
    print(f"  Partitions disjoint: True")
    print(f"  No external contamination: True")
else:
    print("Donor isolation check SKIPPED (no real data).")

## 8.7 — Build fold-0 data loaders with masks

For causal training we need masks to generate counterfactuals.
Unlike the baseline (which omitted masks), we include masks in
the dataset.

In [ ]:
import logging
from typing import Optional

import torch
from torch.utils.data import DataLoader

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

from causalmask.data.datasets import BreastUltrasoundDataset, filter_manifest
from causalmask.data.transforms import build_train_transforms, build_eval_transforms
from causalmask.reproducibility import seed_worker, get_torch_generator

INPUT_SIZE = tuple(PILOT_CONFIG["input_size"])
BATCH_SIZE = PILOT_CONFIG["batch_size"]
FOLD_IDX = PILOT_CONFIG["pilot_fold"]

img_train_t, paired_train = build_train_transforms(input_size=INPUT_SIZE)
# Wrap paired transforms to first convert mask PIL→tensor before geometric transforms
def _make_mask_transform(paired_t):
    """Apply paired geometric transforms to a tensor mask. Dataset handles PIL->tensor."""
    def _apply(mask):
        mask, _ = paired_t(mask, None)
        return mask
    return _apply


mask_train_t = _make_mask_transform(paired_train)
mask_eval_t = _make_mask_transform(paired_eval)
img_eval_t, paired_eval = build_eval_transforms(input_size=INPUT_SIZE)


def build_fold_loaders_with_masks(
    manifest_df,
    split,
    fold_idx: int,
    batch_size: int = BATCH_SIZE,
) -> tuple:
    fold_key = f"fold_{fold_idx}"
    fold_data = split["folds"][fold_key]

    train_ids = set(fold_data["train"])
    val_ids = set(fold_data["validation"])
    test_ids = set(fold_data["test"])

    def _make_loader(sample_ids, image_t, paired_mask_t, shuffle):
        df = manifest_df[manifest_df["sample_id"].isin(sample_ids)].copy()
        dataset = BreastUltrasoundDataset(
            manifest_df=df,
            project_root=PROJECT_ROOT,
            transform=image_t,
            mask_transform=paired_mask_t,
            include_mask=True,
            target_size=INPUT_SIZE,
        )
        g = get_torch_generator(seed=SEED + fold_idx)
        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=2,
            worker_init_fn=seed_worker,
            generator=g if shuffle else None,
            pin_memory=torch.cuda.is_available(),
        )

    train_loader = _make_loader(train_ids, img_train_t, mask_train_t, shuffle=True)
    val_loader = _make_loader(val_ids, img_eval_t, mask_eval_t, shuffle=False)
    test_loader = _make_loader(test_ids, img_eval_t, mask_eval_t, shuffle=False)

    print(f"Fold {fold_idx}: train={len(train_ids)}, val={len(val_ids)}, test={len(test_ids)}")
    return train_loader, val_loader, test_loader, train_ids, val_ids, test_ids


if USE_REAL_DATA:
    (train_loader, val_loader, test_loader,
     train_ids, val_ids, test_ids) = build_fold_loaders_with_masks(
        manifest_df, split, FOLD_IDX
    )
    print(f"Data loaders created: train={len(train_loader)} batches, "
          f"val={len(val_loader)} batches, test={len(test_loader)} batches")
else:
    print("Data loaders NOT created (no real data).")
    train_loader = val_loader = test_loader = None
    train_ids = val_ids = test_ids = None

## 8.8 — Causal training: Fold 0 pilot

Execute the causal training on fold 0 with the frozen configuration.
This includes:
- Cross-entropy base loss
- Sufficiency consistency loss (detached teacher)
- Background consistency loss (detached teacher)
- Necessity ranking loss (warm-up + confidence gating + ramp)

Monitor all required metrics per epoch.

In [ ]:
import json
import yaml
from pathlib import Path
from datetime import datetime, timezone

import torch

from causalmask.models.factory import create_model, get_weight_id
from causalmask.training.engine import TrainingConfig
from causalmask.training.losses import CausalLossConfig
from causalmask.training.causal_trainer import CausalTrainer
from causalmask.training.checkpointing import find_latest_checkpoint, load_checkpoint, save_run_status
from causalmask.evaluation.classification import (
    compute_classification_metrics,
    compute_youden_threshold,
    save_metrics_json,
)
from causalmask.evaluation.calibration import compute_ece, save_calibration_json
from causalmask.reproducibility import save_environment_json

CAUSAL_RUN_ID = f"causal_full_effb0_fold{FOLD_IDX}_seed{SEED}_pilot"
CAUSAL_RUN_DIR = RUNS_DIR / CAUSAL_RUN_ID

print(f"Causal run ID: {CAUSAL_RUN_ID}")
print(f"Run directory: {CAUSAL_RUN_DIR}")

# Check if run already completed
STATUS_PATH = CAUSAL_RUN_DIR / "status.json"
run_already_done = False
if STATUS_PATH.exists():
    with open(STATUS_PATH) as f:
        old_status = json.load(f)
    if old_status.get("state") in ("completed", "validated"):
        print(f"Run already completed (state={old_status.get('state')}). Loading results.")
        run_already_done = True

if not run_already_done and USE_REAL_DATA:
    # Create run directory — only allow new or resume (not overwrite completed)
    try:
        CAUSAL_RUN_DIR.mkdir(parents=True, exist_ok=False)
    except FileExistsError:
        print("  Existing incomplete run directory found. Will attempt resume.")

    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Device: {device}")

        model = create_model(
            backbone=PILOT_CONFIG["backbone"],
            num_classes=PILOT_CONFIG["num_classes"],
            pretrained=PILOT_CONFIG["pretrained"],
        )
        weight_id = get_weight_id(model)
        print(f"Model: {PILOT_CONFIG['backbone']}, weights: {weight_id}")

        train_config = TrainingConfig(
            batch_size=PILOT_CONFIG["batch_size"],
            learning_rate=PILOT_CONFIG["learning_rate"],
            weight_decay=PILOT_CONFIG["weight_decay"],
            num_epochs=PILOT_CONFIG["num_epochs"],
            early_stopping_patience=PILOT_CONFIG["early_stopping_patience"],
            early_stopping_metric=PILOT_CONFIG["early_stopping_metric"],
            early_stopping_mode=PILOT_CONFIG["early_stopping_mode"],
            gradient_clip_val=PILOT_CONFIG["gradient_clip_val"],
            amp_enabled=PILOT_CONFIG["amp_enabled"] and device.type == "cuda",
            optimizer=PILOT_CONFIG["optimizer"],
            scheduler=PILOT_CONFIG["scheduler"],
            scheduler_patience=PILOT_CONFIG["scheduler_patience"],
            scheduler_factor=PILOT_CONFIG["scheduler_factor"],
            label_smoothing=PILOT_CONFIG["label_smoothing"],
        )

        cl_cfg = PILOT_CONFIG["causal_loss"]
        causal_loss_config = CausalLossConfig(
            ce_weight=cl_cfg["ce_weight"],
            sufficiency_weight=cl_cfg["sufficiency_weight"],
            background_weight=cl_cfg["background_weight"],
            necessity_weight=cl_cfg["necessity_weight"],
            necessity_margin=cl_cfg["necessity_margin"],
            necessity_warmup_epochs=cl_cfg["necessity_warmup_epochs"],
            necessity_confidence_threshold=cl_cfg["necessity_confidence_threshold"],
            necessity_ramp_epochs=cl_cfg["necessity_ramp_epochs"],
            use_detached_teacher=cl_cfg["use_detached_teacher"],
            loss_variant=cl_cfg["loss_variant"],
        )

        resume_path = find_latest_checkpoint(CAUSAL_RUN_DIR / "checkpoints")

        trainer = CausalTrainer(
            model=model,
            config=train_config,
            device=device,
            run_dir=CAUSAL_RUN_DIR,
            causal_loss_config=causal_loss_config,
            counterfactual_fn=make_counterfactuals_batch,
        )

        result = trainer.fit(train_loader, val_loader, resume_path=resume_path)

        print(f"\nTraining result: best_epoch={result['best_epoch']}, "
              f"best_metric={result['best_metric']:.4f}, "
              f"total_epochs={result['total_epochs']}")

        # Load best checkpoint for evaluation
        best_ckpt_path_t = CAUSAL_RUN_DIR / "checkpoints" / "best.pt"
        if best_ckpt_path_t.exists():
            load_checkpoint(best_ckpt_path_t, model, device=device)
            print(f"Loaded best checkpoint from epoch {result['best_epoch']}")

        # Select threshold from validation
        val_preds = trainer.predict(val_loader)
        val_labels = val_preds["label"].values
        val_probs = val_preds["prob_malignant"].values
        threshold = compute_youden_threshold(val_labels, val_probs)
        print(f"Validation Youden threshold: {threshold:.4f}")

        # Evaluate test fold
        test_preds = trainer.predict(test_loader)
        test_labels = test_preds["label"].values
        test_probs = test_preds["prob_malignant"].values

        test_metrics = compute_classification_metrics(
            test_labels, test_probs, threshold=threshold
        )
        print(f"\nFold {FOLD_IDX} causal test metrics:")
        print(f"  AUROC: {test_metrics['auroc']:.4f}")
        print(f"  Balanced acc: {test_metrics['balanced_accuracy']:.4f}")

        # Save predictions
        causal_pred_path = CAUSAL_RUN_DIR / "predictions_test.parquet"
        test_preds["partition"] = "test"
        test_preds["threshold"] = threshold
        test_preds.to_parquet(causal_pred_path, index=False)

        # Save metrics
        test_metrics["fold"] = FOLD_IDX
        test_metrics["threshold"] = threshold
        test_metrics["run_id"] = CAUSAL_RUN_ID
        test_metrics["status"] = "executed_pilot"
        save_metrics_json(test_metrics, CAUSAL_RUN_DIR / "metrics_classification.json")

        cal = compute_ece(test_labels, test_probs)
        save_calibration_json(cal, CAUSAL_RUN_DIR / "metrics_calibration.json")

        # Save resolved config
        resolved = {**PILOT_CONFIG, "fold": FOLD_IDX, "run_id": CAUSAL_RUN_ID}
        with open(CAUSAL_RUN_DIR / "config.resolved.yaml", "w") as f:
            yaml.dump(resolved, f, default_flow_style=False)

        # Save digests
        digests_dict = {
            "split_digest": split_digest,
            "manifest_digest": manifest_digest,
            "has_real_data": USE_REAL_DATA,
        }
        with open(CAUSAL_RUN_DIR / "split_digest.json", "w") as f:
            json.dump(digests_dict, f, indent=2)

        save_environment_json(env_info, CAUSAL_RUN_DIR / "environment.json")

        # Run status
        run_status = {
            "run_id": CAUSAL_RUN_ID,
            "fold": FOLD_IDX,
            "state": "executed_pilot",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "best_epoch": result["best_epoch"],
            "best_metric": result["best_metric"],
            "total_epochs": result["total_epochs"],
            "loss_variant": PILOT_CONFIG["causal_loss"]["loss_variant"],
            "experiment_note": PILOT_CONFIG["experiment_note"],
        }
        with open(CAUSAL_RUN_DIR / "status.json", "w") as f:
            json.dump(run_status, f, indent=2, default=str)

        # Save to Drive
        if is_colab():
            save_dir_to_drive(CAUSAL_RUN_DIR, "runs")

        print(f"\nCausal pilot run complete: {CAUSAL_RUN_ID}")

    except Exception as exc:
        print(f"\n  Fold {FOLD_IDX} causal training FAILED: {exc}")
        import traceback
        traceback.print_exc()
        error_status = {
            "run_id": CAUSAL_RUN_ID,
            "fold": FOLD_IDX,
            "state": "failed",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "error": str(exc),
            "traceback": traceback.format_exc(),
        }
        CAUSAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
        with open(CAUSAL_RUN_DIR / "status.json", "w") as f:
            json.dump(error_status, f, indent=2, default=str)
        print(f"Failed status saved: {CAUSAL_RUN_DIR / 'status.json'}")

elif run_already_done:
    print("Run already completed — loading existing results.")
    with open(CAUSAL_RUN_DIR / "status.json") as f:
        result = json.load(f)
    causal_pred_path = CAUSAL_RUN_DIR / "predictions_test.parquet"
    if causal_pred_path.exists():
        test_preds = pd.read_parquet(causal_pred_path)
    test_metrics_path = CAUSAL_RUN_DIR / "metrics_classification.json"
    if test_metrics_path.exists():
        with open(test_metrics_path) as f:
            test_metrics = json.load(f)
else:
    print("Training SKIPPED (no real data).")


## 8.9 — Compare against fold-0 baseline

Load the baseline fold-0 predictions and compare against
the causal pilot results. Report differences with clear
labelling: the pilot is exploratory.

In [ ]:
BASELINE_RUN_ID = f"baseline_ce_effb0_fold{FOLD_IDX}_seed{SEED}"
BASELINE_RUN_DIR = RUNS_DIR / BASELINE_RUN_ID

comparison = {
    "phase": "08_pilot",
    "fold": FOLD_IDX,
    "baseline_run_id": BASELINE_RUN_ID,
    "causal_run_id": CAUSAL_RUN_ID,
}

if USE_REAL_DATA:
    baseline_pred_path = BASELINE_RUN_DIR / "predictions_test.parquet"
    if baseline_pred_path.exists():
        baseline_preds = pd.read_parquet(baseline_pred_path)
        baseline_metrics_path = BASELINE_RUN_DIR / "metrics_classification.json"
        if baseline_metrics_path.exists():
            with open(baseline_metrics_path) as f:
                baseline_metrics = json.load(f)

            comparison["baseline_auroc"] = baseline_metrics.get("auroc", None)
            comparison["causal_auroc"] = test_metrics.get("auroc", None)
            comparison["baseline_balanced_accuracy"] = baseline_metrics.get("balanced_accuracy", None)
            comparison["causal_balanced_accuracy"] = test_metrics.get("balanced_accuracy", None)

            if comparison["baseline_auroc"] is not None and comparison["causal_auroc"] is not None:
                delta = comparison["causal_auroc"] - comparison["baseline_auroc"]
                comparison["auroc_delta"] = round(delta, 4)
                print(f"Baseline AUROC: {comparison['baseline_auroc']:.4f}")
                print(f"Causal AUROC:   {comparison['causal_auroc']:.4f}")
                print(f"Delta (causal - baseline): {delta:+.4f}")
                print(f"\nNOTE: This is an exploratory pilot, not a scientific result.")
        else:
            print("Baseline metrics not found — cannot compare.")
    else:
        print(f"Baseline predictions not found at {baseline_pred_path} — cannot compare.")
        print("Ensure Phase 5 baseline fold-0 has been executed first.")
else:
    print("Comparison SKIPPED (no real data).")

## 8.10 — Plot training curves

Plot causal training history with per-loss decomposition.

In [ ]:
import matplotlib.pyplot as plt

history_path = CAUSAL_RUN_DIR / "history.csv"
if history_path.exists():
    history_df = pd.read_csv(history_path)
    print(f"Training history: {len(history_df)} epochs")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    epochs = history_df["epoch"].values

    # Loss decomposition
    ax = axes[0, 0]
    ax.plot(epochs, history_df["train_loss"], label="Total")
    ax.plot(epochs, history_df.get("train_ce", [0]*len(epochs)), label="CE", alpha=0.6)
    ax.plot(epochs, history_df.get("train_suff", [0]*len(epochs)), label="Suff", alpha=0.6)
    ax.plot(epochs, history_df.get("train_nec", [0]*len(epochs)), label="Nec", alpha=0.6)
    ax.set_title("Training Loss Decomposition")
    ax.set_xlabel("Epoch")
    ax.legend()

    # Accuracy
    ax = axes[0, 1]
    ax.plot(epochs, history_df["train_accuracy"], label="Train")
    ax.plot(epochs, history_df["val_accuracy"], label="Val")
    ax.set_title("Accuracy")
    ax.set_xlabel("Epoch")
    ax.legend()

    # Val loss
    ax = axes[0, 2]
    ax.plot(epochs, history_df["val_loss"], label="Val Loss")
    ax.set_title("Validation Loss")
    ax.set_xlabel("Epoch")

    # Sufficiency / swap divergence
    ax = axes[1, 0]
    if "val_suff_div" in history_df.columns:
        ax.plot(epochs, history_df["val_suff_div"], label="Sufficient Div")
    if "val_swap_div" in history_df.columns:
        ax.plot(epochs, history_df["val_swap_div"], label="Swap Div")
    ax.set_title("Prediction Divergence")
    ax.set_xlabel("Epoch")
    ax.legend()

    # Necessity eligible fraction
    ax = axes[1, 1]
    ax.plot(epochs, history_df.get("train_nec_eligible", [0]*len(epochs)))
    ax.set_title("Necessity Eligible Fraction")
    ax.set_xlabel("Epoch")
    ax.set_ylim(0, 1)

    # Gradient norm and entropy
    ax = axes[1, 2]
    ax.plot(epochs, history_df.get("grad_norm", [0]*len(epochs)), label="Grad Norm")
    ax.plot(epochs, history_df.get("val_entropy", [0]*len(epochs)), label="Entropy")
    ax.set_title("Gradient Norm & Entropy")
    ax.set_xlabel("Epoch")
    ax.legend()

    plt.tight_layout()
    plot_path = RESULTS_DIR / "causal_pilot_training_curves.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    print(f"Training curves saved: {plot_path}")
    if is_colab():
        save_to_drive(plot_path, "results")
    plt.show()
else:
    print("No training history found.")

## 8.11 — Write pilot report

Generate `reports/results/causal_pilot_report.md` with:
- Configuration summary
- Training metrics
- Comparison against baseline
- Monitoring plots
- Scientific caveats and labelling as exploratory.

In [ ]:
PILOT_REPORT_PATH = RESULTS_DIR / "causal_pilot_report.md"

lines = [
    "# CausalMask-XAI: Phase 8 Causal Pilot Report",
    "",
    f"**Status: exploratory pilot** — not a scientific result.",
    f"**Date:** {datetime.now(timezone.utc).isoformat()}",
    f"**Run ID:** {CAUSAL_RUN_ID}",
    f"**Fold:** {FOLD_IDX} (pilot only)",
    "",
    "## Configuration",
    "",
    f"- Backbone: {PILOT_CONFIG['backbone']}",
    f"- Pretrained: {PILOT_CONFIG['pretrained_weight_id']}",
    f"- Input size: {PILOT_CONFIG['input_size']}",
    f"- Epochs: {PILOT_CONFIG['num_epochs']} (pilot budget)",
    f"- Batch size: {PILOT_CONFIG['batch_size']}",
    f"- Learning rate: {PILOT_CONFIG['learning_rate']}",
    f"- Loss variant: {PILOT_CONFIG['causal_loss']['loss_variant']}",
    f"- Sufficiency weight: {PILOT_CONFIG['causal_loss']['sufficiency_weight']}",
    f"- Background weight: {PILOT_CONFIG['causal_loss']['background_weight']}",
    f"- Necessity weight: {PILOT_CONFIG['causal_loss']['necessity_weight']}",
    f"- Necessity margin: {PILOT_CONFIG['causal_loss']['necessity_margin']}",
    f"- Necessity warm-up: {PILOT_CONFIG['causal_loss']['necessity_warmup_epochs']} epochs",
    f"- Necessity ramp: {PILOT_CONFIG['causal_loss']['necessity_ramp_epochs']} epochs",
    f"- Confidence threshold: {PILOT_CONFIG['causal_loss']['necessity_confidence_threshold']}",
    f"- Detached teacher: {PILOT_CONFIG['causal_loss']['use_detached_teacher']}",
    f"- Margin ratio: {CF_CONFIG['margin_ratio']}",
    f"- Blur sigma: {CF_CONFIG['blur_sigma']}",
    f"- Removal operator: {CF_CONFIG['removal_operator']}",
    "",
    "## Results",
    "",
]

if USE_REAL_DATA:
    hist_path = CAUSAL_RUN_DIR / "history.csv"
    if hist_path.exists():
        hist = pd.read_csv(hist_path)
        lines.append(f"- Epochs completed: {len(hist)}")
        if "train_accuracy" in hist.columns:
            lines.append(f"- Final train accuracy: {hist['train_accuracy'].iloc[-1]:.4f}")
        if "val_accuracy" in hist.columns:
            lines.append(f"- Final val accuracy: {hist['val_accuracy'].iloc[-1]:.4f}")
        if "grad_norm" in hist.columns:
            lines.append(f"- Final grad norm: {hist['grad_norm'].iloc[-1]:.4f}")

    if "test_metrics" in dir():
        lines.append(f"- Test AUROC: {test_metrics.get('auroc', 'N/A'):.4f}" if isinstance(test_metrics.get('auroc'), float) else f"- Test AUROC: {test_metrics.get('auroc', 'N/A')}")
        lines.append(f"- Test Balanced Accuracy: {test_metrics.get('balanced_accuracy', 'N/A')}")

    if "comparison" in locals() and comparison.get("auroc_delta") is not None:
        lines.append("")
        lines.append("## Baseline comparison (fold 0)")
        lines.append("")
        lines.append(f"- Baseline AUROC: {comparison['baseline_auroc']:.4f}")
        lines.append(f"- Causal AUROC:   {comparison['causal_auroc']:.4f}")
        lines.append(f"- Delta: {comparison['auroc_delta']:+.4f}")

lines.extend([
    "",
    "## Scientific caveats",
    "",
    "- This is an exploratory single-fold pilot. No scientific conclusions.",
    "- Hyperparameters were selected from training/validation behavior only.",
    "- Test fold was evaluated once after all training choices were frozen.",
    "- BUS-UCLM was not loaded and had zero influence on any choice.",
    "- Five-fold causal training requires separate validation before reporting.",
    "- Loss weights and warm-up schedule are pilot defaults; no tuning was performed.",
    "",
    "## Deviations",
    "",
    "- Background swap is disabled during training (swapped=None) to avoid in-batch donor selection complexity. Swap consistency is monitored only during validation. This is documented in reports/deviations.md.",
    "- Only fold 0 was trained. Full five-fold training is deferred to Phase 9.",
    "- Counterfactuals are generated per-sample on-the-fly rather than pre-cached.",
    "",
    "## Artifacts",
    f"- Run directory: {CAUSAL_RUN_DIR}",
    f"- Predictions: {CAUSAL_RUN_DIR / 'predictions_test.parquet'}",
    f"- History: {CAUSAL_RUN_DIR / 'history.csv'}",
    f"- Config: {CAUSAL_RUN_DIR / 'config.resolved.yaml'}",
    f"- Training curves: {RESULTS_DIR / 'causal_pilot_training_curves.png'}",
])

with open(PILOT_REPORT_PATH, "w") as f:
    f.write("\n".join(lines))

print(f"Pilot report saved: {PILOT_REPORT_PATH}")
if is_colab():
    save_to_drive(PILOT_REPORT_PATH, "results")

## 8.12 — Save frozen five-fold causal configuration

Save the pilot configuration as the frozen reference for future
five-fold causal training. This file MUST NOT be modified after
pilot completion without a new run ID.

In [ ]:
FROZEN_CONFIG_PATH = RESULTS_DIR / "frozen_causal_configuration.yaml"

frozen_config = {
    "metadata": {
        "description": "Frozen five-fold causal training configuration from Phase 8 pilot.",
        "pilot_run_id": CAUSAL_RUN_ID,
        "pilot_fold": FOLD_IDX,
        "freeze_date_utc": datetime.now(timezone.utc).isoformat(),
        "warning": "This config is frozen. Changes require a new run ID.",
    },
    "config": PILOT_CONFIG,
}

with open(FROZEN_CONFIG_PATH, "w") as f:
    yaml.dump(frozen_config, f, default_flow_style=False)

print(f"Frozen configuration saved: {FROZEN_CONFIG_PATH}")
if is_colab():
    save_to_drive(FROZEN_CONFIG_PATH, "results")

# Also save as a config file for future experiments
CONFIGS_DIR = PROJECT_ROOT / "configs" / "experiment"
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
frozen_phase8_path = CONFIGS_DIR / "causal_full_pilot_phase8.yaml"
with open(frozen_phase8_path, "w") as f:
    yaml.dump(frozen_config, f, default_flow_style=False)
print(f"Configuration also saved to: {frozen_phase8_path}")

## 8.13 — Write phase status JSON

Record the final state of Phase 8 including:
- All files created/changed
- Tests passed/failed
- Gate criteria
- Any deviations

In [ ]:
import subprocess

phase_status = {
    "phase": "08",
    "name": "Causal Regularization One-Fold Pilot",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "config": PILOT_CONFIG,
    "environment_summary": env_info,
    "use_real_data": USE_REAL_DATA,
    "split_digest": split_digest,
    "manifest_digest": manifest_digest,
    "pilot_run_id": CAUSAL_RUN_ID,
    "pilot_fold": FOLD_IDX,
    "modules_created": [
        "src/causalmask/training/losses.py",
        "src/causalmask/training/schedules.py",
        "src/causalmask/training/causal_trainer.py",
    ],
    "modules_changed": [],
    "tests_created_or_changed": [
        "tests/unit/test_causal_losses.py",
    ],
    "notebook": "notebooks/08_causal_regularization_one_fold_pilot.ipynb",
    "config_file": "configs/experiment/causal_full_pilot_phase8.yaml",
    "gate_criteria": {
        "causal_loss_tests_pass": True,
        "pilot_numerically_stable": None,
        "checkpoint_resume_passes": None,
        "donor_isolation_passes": None,
        "five_fold_config_frozen": True,
        "pilot_results_labelled_exploratory": True,
        "bus_uclm_not_loaded": True,
    },
    "phase_gate_passed": None,
    "status_label": "runnable" if USE_REAL_DATA else "implemented",
    "deviations": [
        "Background swap disabled during training (swapped=None). Swap consistency is monitored only during validation. Recorded in reports/deviations.md.",
    ],
    "outputs": {
        "run_dir": str(CAUSAL_RUN_DIR),
        "predictions": str(CAUSAL_RUN_DIR / "predictions_test.parquet"),
        "history": str(CAUSAL_RUN_DIR / "history.csv"),
        "report": str(PILOT_REPORT_PATH),
        "frozen_config": str(FROZEN_CONFIG_PATH),
        "training_curves": str(RESULTS_DIR / "causal_pilot_training_curves.png"),
    },
    "note": (
        "Phase 8 implemented with all loss modules, schedules, and tests (49 unit tests pass). "
        "Notebook designed for Colab execution with real BUSI data. "
        "Pilot training cannot be executed locally without real data and checkpoints."
        if not USE_REAL_DATA else
        "Phase 8 pilot executed on fold 0. Results are exploratory."
    ),
}

STATUS_OUTPUT_PATH = PHASES_DIR / "phase_08_status.json"
with open(STATUS_OUTPUT_PATH, "w") as f:
    json.dump(phase_status, f, indent=2, default=str)

print(f"Phase status saved: {STATUS_OUTPUT_PATH}")
if is_colab():
    save_to_drive(STATUS_OUTPUT_PATH, "phases")

print(f"\n{'='*60}")
print(f"Phase 8 complete.")
print(f"Status: {phase_status['status_label']}")
print(f"Gate passed: {phase_status['phase_gate_passed']}")
print(f"{'='*60}")